In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
import json
from datetime import datetime

In [ ]:
# CẤU HÌNH 
base_url = "https://vncooking.com/cong-thuc"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

all_links = []
    
# CHỈNH SỐ TRANG CẦN CÀO
start_page = 1
end_page = 46  
    
print(f"BẮT ĐẦU CÀO LINK ")

for page in range(start_page, end_page + 1):
    try:
        url = base_url if page == 1 else f"{base_url}?p={page}"
        print(f"Đang quét trang danh sách: {url}")
            
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            items = soup.find_all('a', class_='recipe-index-item')
            for item in items:
                link = item.get('href')
                if link:
                    if not link.startswith('http'):
                        link = "https://vncooking.com" + link
                    all_links.append(link)
            
        time.sleep(1) # Nghỉ nhẹ tránh spam
    except Exception as e:
        print(f"Lỗi trang {page}: {e}")

print(f"\n-> Tổng link tìm được: {len(all_links)}")

=== BẮT ĐẦU CÀO LINK (Trang 1 -> 46) ===
Đang quét trang danh sách: https://vncooking.com/cong-thuc
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=2
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=3
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=4
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=5
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=6
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=7
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=8
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=9
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=10
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=11
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=12
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=13
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=14
Đang quét trang danh sách: https://vncooking.com/cong-thuc?p=15
Đang quét tr

In [6]:
# HÀM CÀO CHI TIẾT
def get_recipe_detail(url):
    data = {
        "link": url,
        "type_of_food": None, "title": None, "description": None,
        "author_name": None, "cook_time": None, "num_of_people": None,
        "calories": None, "num_of_ingredients": None,
        "ingredients": [],
        "step": [],        
        "note": [],        
        "post_date": None,
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code != 200:
            print(f"   -> Lỗi tải trang: {response.status_code}")
            return data

        soup = BeautifulSoup(response.content, 'html.parser')

        # BÓC TÁCH DỮ LIỆU CƠ BẢN 
        if title := soup.find('h1', id='recipe-content-detail-title'):
            data['title'] = title.get_text(strip=True)
            
        if desc := soup.find('div', id='recipe-content-detail-description'):
            data['description'] = desc.get_text(strip=True)
            
        if breadcrumb := soup.find('div', id='breadcrumb'):
            links = breadcrumb.find_all('a')
            if len(links) >= 3: data['type_of_food'] = links[2].get_text(strip=True)

        # INFO BLOCK 
        info_block = soup.find('div', id='recipe-content-detail-info')
        if info_block:
            items = info_block.find_all('div', class_='recipe-content-detail-info-item')
            for item in items:
                name_tag = item.find('div', class_='recipe-content-detail-info-item-name')
                value_tag = item.find('div', class_='recipe-content-detail-info-item-value')
                if name_tag and value_tag:
                    name_text = name_tag.get_text(strip=True).lower()
                    value_text = value_tag.get_text(strip=True)
                    if 'nguyên liệu' in name_text: data['num_of_ingredients'] = value_text
                    elif 'số người' in name_text: data['num_of_people'] = value_text
                    elif 'thời gian' in name_text: data['cook_time'] = value_text
                    elif 'calories' in name_text or 'calo' in name_text: data['calories'] = value_text

        # INGREDIENTS 
        if ing_div := soup.find('div', id='recipe-content-detail-ingredients'):
            clean_ings = []
            
            ing_items = ing_div.find_all('div', class_='ingredient')
            
            if ing_items:
                for item in ing_items:
                    name_tag = item.find('div', class_='ingredient-name')
                    if name_tag:
                        full_text = name_tag.get_text(' ', strip=True)
                        full_text = full_text.replace(',', '')      
                        full_text = re.sub(r'\s+', ' ', full_text) 
                        clean_ings.append(full_text)
            else:
                 raw_items = ing_div.find_all(['li', 'p'])
                 if raw_items:
                    for item in raw_items:
                        text = item.get_text(' ', strip=True).replace(',', '')
                        if text and "nguyên liệu" not in text.lower():
                             clean_ings.append(re.sub(r'\s+', ' ', text))
            
            data['ingredients'] = clean_ings

        if step_div := soup.find('div', id='recipe-content-detail-steps'):
            processed_steps = []
            step_blocks = step_div.find_all('div', class_='recipe-step')
            
            if step_blocks:
                for block in step_blocks:
                    num_tag = block.find('div', class_='recipe-step-number')
                    desc_tag = block.find('div', class_='recipe-step-description')
                    
                    if desc_tag:
                        content = desc_tag.get_text(' ', strip=True)
                        step_num = "0"
                        if num_tag:
                            nums = re.findall(r'\d+', num_tag.get_text(strip=True))
                            if nums: step_num = str(int(nums[0])) # 01 -> 1
                        
                        processed_steps.append(f"Bước {step_num}: {content}")

            if not processed_steps:
                 raw_text = step_div.get_text('\n', strip=True)
                 for line in raw_text.split('\n'):
                     if line.strip() and "thực hiện" not in line.lower():
                         processed_steps.append(line.strip())

            data['step'] = processed_steps

        # POST DATE 
        date_pattern = re.compile(r'(Thứ|Chủ)\s+.*?\d{1,2}/\d{1,2}/\d{4}.*?\(GMT\+7\)', re.IGNORECASE)
        found_date = soup.find(string=date_pattern)
        
        if found_date:
            data['post_date'] = found_date.strip()
        else:
            if meta_date := soup.find('meta', property='article:published_time'):
                raw_date = meta_date.get('content') 
                try:
                    dt = datetime.fromisoformat(raw_date)
                    days_vi = {
                        0: "Thứ hai", 1: "Thứ ba", 2: "Thứ tư", 3: "Thứ năm",
                        4: "Thứ sáu", 5: "Thứ bảy", 6: "Chủ nhật"
                    }
                    weekday_str = days_vi[dt.weekday()]
                    formatted_date = f"{weekday_str}, {dt.day}/{dt.month}/{dt.year}, {dt.strftime('%H:%M')} (GMT+7)"
                    data['post_date'] = formatted_date
                except Exception:
                    data['post_date'] = raw_date

    except Exception as e:
        print(f"   -> Lỗi ngoại lệ: {e}")
    
    return data

# CÀO CHI TIẾT TỪNG MÓN 
final_data = []
print(f"\n=== BẮT ĐẦU CÀO CHI TIẾT ===")

# Duyệt qua từng link
for idx, link in enumerate(all_links): 
    print(f"[{idx+1}/{len(all_links)}] Đang xử lý: {link}")
    details = get_recipe_detail(link)
    final_data.append(details)
        
    # Nghỉ 1 giây để an toàn
    time.sleep(1)


=== BẮT ĐẦU CÀO CHI TIẾT ===
[1/544] Đang xử lý: https://vncooking.com/cong-thuc/tom-hap-lien-hoa-559
[2/544] Đang xử lý: https://vncooking.com/cong-thuc/tra-lipton-pha-voi-7up-558
[3/544] Đang xử lý: https://vncooking.com/cong-thuc/cach-lam-kem-sau-rieng-thom-ngon-nhanh-cap-toc-khong-can-tu-lanh-557
[4/544] Đang xử lý: https://vncooking.com/cong-thuc/cach-lam-bun-ga-la-e-ngot-thom-chua-thanh-cho-ca-gia-dinh-556
[5/544] Đang xử lý: https://vncooking.com/cong-thuc/khoai-tay-nghien-pho-mai-cap-toc-bang-snack-khoai-tay-555
[6/544] Đang xử lý: https://vncooking.com/cong-thuc/banh-flan-duong-thot-not-mem-min-thom-lung-554
[7/544] Đang xử lý: https://vncooking.com/cong-thuc/banh-bao-bo-xanh-muot-mem-thom-beo-ngay-553
[8/544] Đang xử lý: https://vncooking.com/cong-thuc/banh-su-mini-sieu-de-beo-ngay-bat-ngo-552
[9/544] Đang xử lý: https://vncooking.com/cong-thuc/trung-luoc-7-sao-dubai-doc-la-trung-beo-thom-dam-da-551
[10/544] Đang xử lý: https://vncooking.com/cong-thuc/goi-tom-rau-nhut-gion-t

In [9]:
# LƯU KẾT QUẢ RA EXCEL 

print("\n=== ĐANG LƯU FILE EXCEL ===")
        
df = pd.DataFrame(final_data)

# Chuyển List thành String 
df['ingredients'] = df['ingredients'].apply(lambda x: '\n'.join(x) if isinstance(x, list) else x)
df['step'] = df['step'].apply(lambda x: '\n'.join(x) if isinstance(x, list) else x)
df['note'] = df['note'].apply(lambda x: '\n'.join(x) if isinstance(x, list) else x)

# Sắp xếp cột theo đúng thứ tự
cols = ['link', 'type_of_food', 'title', 'description', 'author_name', 
                'cook_time', 'num_of_people', 'calories', 'num_of_ingredients', 
                'ingredients', 'step', 'note', 'post_date']
        
# Reindex đảm bảo đủ cột dù data thiếu
df = df.reindex(columns=cols)

output_file = 'vncooking_final_data.xlsx'
df.to_excel(output_file, index=False)
        
print(f"HOÀN THÀNH! File đã lưu tại: {output_file}")



=== ĐANG LƯU FILE EXCEL ===
HOÀN THÀNH! File đã lưu tại: vncooking_final_data.xlsx


In [8]:
df.head()

,link,type_of_food,title,description,author_name,cook_time,num_of_people,calories,num_of_ingredients,ingredients,step,note,post_date
0,https://vncooking.com/cong-thuc/tom-hap-lien-h...,Món chính,Tôm hấp liên hoa,Tôm hấp liên hoa không chỉ có hình thức bắt mắ...,None,30phút,2,None,3,tôm thẻ 200 gam\ngiò sống 500 gam\nHành tây 1 củ,"Bước 1: Sơ chế nguyên liệu Tôm mua về bóc vỏ, ...",,"Thứ năm, 19/10/2023, 20:08 (GMT+7)"
1,https://vncooking.com/cong-thuc/tra-lipton-pha...,Thức uống,Trà Lipton pha với 7up,Đã bao giờ bạn thử món nước giải khát được làm...,None,15phút,2,None,2,trà Lipton 1 túi\n7up 1 chai,"Bước 1: Pha chế trà Lipton với 7up Đầu tiên, b...",,"Thứ hai, 16/10/2023, 20:11 (GMT+7)"
2,https://vncooking.com/cong-thuc/cach-lam-kem-s...,Bánh và bánh ngọt,Cách làm kem sầu riêng thơm ngon nhanh cấp tốc...,Bạn đã biết cách làm kem sầu riêng thơm ngon g...,None,30phút,2,None,4,Thịt sầu riêng 100 gam\nSữa đặc 50 gam\nSữa tư...,"Bước 1: Sơ chế nguyên liệu Đầu tiên, bạn cho p...",,"Chủ nhật, 15/10/2023, 22:15 (GMT+7)"
3,https://vncooking.com/cong-thuc/cach-lam-bun-g...,Món chính,"Cách làm bún gà lá é ngọt thơm, chua thanh cho...",xin giới thiệu đến bạn món bún gà lá é với hươ...,None,45phút,2,None,3,thịt gà ta 800 gam\nlá é 400 gam\nBún 300 gam,Bước 1: Sơ chế nguyên liệu Thịt gà mua về bạn ...,,"Chủ nhật, 8/10/2023, 20:32 (GMT+7)"
4,https://vncooking.com/cong-thuc/khoai-tay-nghi...,Bánh và bánh ngọt,Khoai tây nghiền phô mai cấp tốc bằng snack kh...,Bạn có thể tự làm khoai tây nghiền phô mai tại...,None,15phút,2,None,2,Snack khoai tây 1 ống\nSữa tươi không đường 1 hộp,"Bước 1: Sơ chế nguyên liệu Đầu tiên, bạn hãy c...",,"Thứ năm, 5/10/2023, 22:08 (GMT+7)"
